In [25]:
# general imports
import pandas as pd
import os
import json

In [ ]:
# Optional environment setup.
# Set these before initializing vLLM, PyTorch, sentence-transformers, or transformers models.

# Restrict this notebook process to specific GPUs.
# Examples: "0" for one GPU, "0,1" for two GPUs.
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Optional: set for higher Hugging Face Hub rate limits and faster first-time downloads.
# os.environ["HF_TOKEN"] = "HF..."

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [ ]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)

(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704] AsyncLLM output_handler failed.
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704] Traceback (most recent call last):
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704]   File "/home/sabrina/mmai-package/.venv/lib/python3.13/site-packages/vllm/v1/engine/async_llm.py", line 660, in output_handler
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704]     outputs = await engine_core.get_output_async()
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704]               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704]   File "/home/sabrina/mmai-package/.venv/lib/python3.13/site-packages/vllm/v1/engine/core_client.py", line 1030, in get_output_async
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704]     raise self._format_exception(outputs) from None
(APIServer pid=107827) ERROR 06-03 21:52:22 [async_llm.py:704] vllm.v1.engine

(APIServer pid=107827) INFO:     Shutting down
(APIServer pid=107827) INFO:     Waiting for application shutdown.
(APIServer pid=107827) INFO:     Application shutdown complete.
(APIServer pid=107827) INFO:     Finished server process [107827]
/home/sabrina/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/multiprocessing/resource_tracker.py:400: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown: {'/loky-108265-v50y1aet'}
  warnings.warn(


### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [28]:
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

INFO 06-03 21:52:45 [utils.py:278] non-default args: {'max_model_len': 30000, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'language_model_only': True, 'model': 'google/gemma-4-31B-it'}
INFO 06-03 21:52:45 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-03 21:52:45 [model.py:1752] Using max model len 30000
INFO 06-03 21:52:51 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-03 21:52:51 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-03 21:52:51 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-03 21:52:51 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-03 21:52:51 [cuda.py:243] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:12<00:12, 12.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  7.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.11s/it]
(EngineCore pid=117400) 


(EngineCore pid=117400) INFO 06-03 21:53:31 [default_loader.py:397] Loading weights took 16.29 seconds
(EngineCore pid=117400) INFO 06-03 21:53:32 [gpu_model_runner.py:5132] Model loading took 57.91 GiB memory and 17.438313 seconds
(EngineCore pid=117400) INFO 06-03 21:53:40 [backends.py:1089] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/feaa32eb45/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=117400) INFO 06-03 21:53:40 [backends.py:1148] Dynamo bytecode transform time: 7.03 s
(EngineCore pid=117400) INFO 06-03 21:53:52 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 11.871 s
(EngineCore pid=117400) INFO 06-03 21:53:52 [decorators.py:311] Directly load AOT compilation from path /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/d534beee4da5fd7f6f9f85e08203882408b42b9b2a29243e0663251cb283e805/rank_0_0/model
(EngineCore pid=117400) INFO 06-03 21:53:52 [monitor.py:53] torch.compile to

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:07<00:00,  6.81it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.19it/s]


(EngineCore pid=117400) INFO 06-03 21:54:12 [gpu_model_runner.py:6456] Graph capturing finished in 13 secs, took 0.82 GiB
(EngineCore pid=117400) INFO 06-03 21:54:12 [gpu_worker.py:619] CUDA graph pool memory: 0.82 GiB (actual), 0.84 GiB (estimated), difference: 0.02 GiB (2.1%).
(EngineCore pid=117400) INFO 06-03 21:54:12 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=117400) INFO 06-03 21:54:12 [core.py:302] init engine (profile, create kv cache, warmup model) took 40.16 s (compilation: 19.66 s)
(EngineCore pid=117400) INFO 06-03 21:54:13 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


Rendering prompts:   2%|▏         | 10/470 [00:00<00:04, 98.81it/s]

(EngineCore pid=117400) WARNING 06-03 21:54:13 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=117400) WARNING 06-03 21:54:13 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 470/470 [3:12:41<00:00, 24.60s/it, est. speed input: 106.21 toks/s, output: 150.79 toks/s]  


### Remote mode

Run this cell instead of the local-mode cell if you want to send trial summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.

In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.trials import summarize_trials

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# process = start_vllm_server(task="trial")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
# process.terminate()

### View results

In [30]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,1,Age range allowed: 60 years or older. Sex allo...,Poor performance status (ECOG > 2). Liver dysf...,NCT03319901-1
1,NCT03319901,2,Age range allowed: 60 years or older. Sex allo...,Poor performance status (ECOG > 2). Liver dysf...,NCT03319901-2
2,NCT03319901,3,Age range allowed: 18 years or older. Sex allo...,Poor performance status (ECOG > 2). Liver dysf...,NCT03319901-3
3,NCT03319901,4,Age range allowed: 18 years or older. Sex allo...,Poor performance status (ECOG > 2). Liver dysf...,NCT03319901-4
4,NCT04792489,1,Age range allowed: 18 years or older. Sex allo...,Poor performance status (eg ECOG >1 unless dec...,NCT04792489-1


Metadata contains a snapshot of the config used and metadata on the model used. 

In [31]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'google/gemma-4-31B-it',
   'reasoning_parser': 'auto',
   'chat_template_kwargs': {'enable_thinking': True},
   'sampling_params': {'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 20,
    'min_p': 0.0,
    'presence_penalty': 1.5,
    'max_tokens': 20000,
    'repetition_penalty': 1.0,
    'skip_special_tokens': False},
   'prompt_files': {'primer'

QC report on the trial summarization run

In [32]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,0.0,470.0,0.000000,[]
1,trials_truncated_llm_response,0.0,470.0,0.000000,[]
2,spaces_exceed_embedding_token_limit,0.0,1927.0,0.000000,[]
3,spaces_per_trial_min,1.0,NaN,NaN,[]
4,spaces_per_trial_median,2.0,NaN,NaN,[]
5,spaces_per_trial_max,42.0,NaN,NaN,[]
6,trials_with_non_distinct_spaces,3.0,470.0,0.638298,"[NCT05652686, NCT06239467, NCT06704724]"
7,spaces_dropped_missing_keyword:Age,0.0,1927.0,0.000000,[]
8,spaces_dropped_missing_keyword:Sex,0.0,1927.0,0.000000,[]
9,spaces_dropped_missing_keyword:Cancer type,0.0,1927.0,0.000000,[]


Save checked-in trial example artifacts


In [33]:
trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in initial notes

In [35]:
notes_path = "data/note_set_1.csv"
notes = pd.read_csv(notes_path)

### Local mode

`summarize_patients(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [36]:
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

/home/sabrina/mmai-package/src/matchminer_ai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


INFO 06-04 01:24:08 [utils.py:278] non-default args: {'max_model_len': 34208, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'language_model_only': True, 'model': 'google/gemma-4-31B-it'}
INFO 06-04 01:24:08 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-04 01:24:08 [model.py:1752] Using max model len 34208
INFO 06-04 01:24:08 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-04 01:24:08 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-04 01:24:08 [cuda.py:243] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.
(EngineCore pid=157790) INFO 06-04 01:24:24 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='google/gemma-4-31B-it', speculative_config=None, toke

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:12<00:12, 12.49s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  7.28s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.06s/it]
(EngineCore pid=157790) 


(EngineCore pid=157790) INFO 06-04 01:24:49 [default_loader.py:397] Loading weights took 16.20 seconds
(EngineCore pid=157790) INFO 06-04 01:24:50 [gpu_model_runner.py:5132] Model loading took 57.91 GiB memory and 17.355291 seconds
(EngineCore pid=157790) INFO 06-04 01:24:57 [backends.py:1089] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/76884c0395/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=157790) INFO 06-04 01:24:57 [backends.py:1148] Dynamo bytecode transform time: 7.03 s
(EngineCore pid=157790) INFO 06-04 01:25:01 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.737 s
(EngineCore pid=157790) INFO 06-04 01:25:01 [decorators.py:311] Directly load AOT compilation from path /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/f70134b0046169638aed928c4357c3f46ef02ab7c2e1ee3e804b854ad439a1ef/rank_0_0/model
(EngineCore pid=157790) INFO 06-04 01:25:01 [monitor.py:53] torch.compile too

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:07<00:00,  6.93it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.37it/s]


(EngineCore pid=157790) INFO 06-04 01:25:20 [gpu_model_runner.py:6456] Graph capturing finished in 13 secs, took 0.82 GiB
(EngineCore pid=157790) INFO 06-04 01:25:20 [gpu_worker.py:619] CUDA graph pool memory: 0.82 GiB (actual), 0.84 GiB (estimated), difference: 0.02 GiB (2.1%).
(EngineCore pid=157790) INFO 06-04 01:25:20 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=157790) INFO 06-04 01:25:20 [core.py:302] init engine (profile, create kv cache, warmup model) took 30.00 s (compilation: 10.52 s)
(EngineCore pid=157790) INFO 06-04 01:25:21 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


Rendering prompts:  17%|█▋        | 2/12 [00:00<00:00, 18.75it/s]

(EngineCore pid=157790) WARNING 06-04 01:25:21 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=157790) WARNING 06-04 01:25:21 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 12/12 [15:27<00:00, 77.27s/it, est. speed input: 220.94 toks/s, output: 35.75 toks/s]


### Remote mode

Run this cell instead of the local-mode cell if you want to send patient summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.


In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.patients import summarize_patients

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# process = start_vllm_server(task="patient")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

patient_summaries, metadata, qc_report = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
#process.terminate()

### Updating existing patient summaries

Run this optional cell after the local- or remote-mode cell above if you want to simulate a longitudinal update. It reads a second set of new notes, uses the existing `patient_summaries` as the starting point, then updates those summaries with the new notes.


In [37]:
from matchminer_ai.patients import summarize_patients

new_notes_path = "data/note_set_2.csv"
new_notes = pd.read_csv(new_notes_path)

existing_summaries = patient_summaries.assign(
    patient_summary=lambda df: (
        df["cancer_history_summary"].fillna("")
        + "\n\nBoilerplate:\n"
        + df["general_exclusion_criteria_evidence"].fillna("")
    )
)[["patient_id", "patient_summary"]]

patient_summaries, metadata, qc_report = summarize_patients(
    new_notes,
    existing_summaries=existing_summaries,
    return_metadata=True,
    return_qc=True,
)

Processed prompts: 100%|██████████| 12/12 [10:43<00:00, 53.63s/it, est. speed input: 220.35 toks/s, output: 54.60 toks/s]


### View results

In [38]:
patient_summaries.head()

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,11609,2026-05-03,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...
1,12019,2026-05-14,Age: 52\nSex: Female\nCancer type: Breast canc...,"Hypertension, hyperlipidemia, Grade 2 ribocicl..."
2,15687,2025-10-07,Age: 68\nSex: Male\nCancer type: Lung cancer\n...,ECOG 1-2. Hypertension. Hyperlipidemia. Osteop...
3,16916000000,2024-11-14,Age: 21\nSex: Male\nCancer type: Osteosarcoma\...,ECOG 0-1. History of Grade 2 immune-mediated c...
4,1722,2025-02-10,Age: 64\nSex: Male\nCancer type: Gastro-esopha...,ECOG 1. Hypertension. Hyperlipidemia. Former s...


Metadata contains a snapshot of the config used and metadata on the models used. 

In [39]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'google/gemma-4-31B-it',
   'reasoning_parser': 'auto',
   'chat_template_kwargs': {'enable_thinking': True},
   'sampling_params': {'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 20,
    'min_p': 0.0,
    'presence_penalty': 1.5,
    'max_tokens': 20000,
    'repetition_penalty': 1.0,
    'skip_special_tokens': False},
   'prompt_files': {'primer'

QC report on patient summarization

In [40]:
qc_report

,metric,value,denominator,percent,ids
0,patients_dropped_noninformative_summary,0,12,0.0,[]
1,patients_exceed_embedding_token_limit,0,12,0.0,[]
2,patients_exclusion_criteria_not_extracted,0,12,0.0,[]
3,patients_missing_keyword:Age,0,12,0.0,[]
4,patients_missing_keyword:Sex,0,12,0.0,[]
5,patients_missing_keyword:Cancer type,0,12,0.0,[]
6,patients_missing_keyword:Histology,0,12,0.0,[]
7,patients_missing_keyword:Current extent,0,12,0.0,[]
8,patients_missing_keyword:Biomarkers,0,12,0.0,[]
9,patients_missing_keyword:Treatment history,0,12,0.0,[]


Save checked-in patient example artifacts

In [41]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [42]:
from matchminer_ai.embedding import embed_for_matching

In [43]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [44]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

In [45]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-1,"[-0.0206298828125, -0.0228271484375, 0.0039978..."
1,NCT03319901-2,"[0.007598876953125, 0.021240234375, 0.00927734..."
2,NCT03319901-3,"[0.000537872314453125, -0.00872802734375, 0.00..."
3,NCT03319901-4,"[0.037841796875, 0.040283203125, 0.01245117187..."
4,NCT04792489-1,"[-0.0260009765625, -0.04296875, 0.00390625, 0...."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [46]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'google/gemma-4-31B-it',
   'reasoning_parser': 'auto',
   'chat_template_kwargs': {'enable_thinking': True},
   'sampling_params': {'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 20,
    'min_p': 0.0,
    'presence_penalty': 1.5,
    'max_tokens': 20000,
    'repetition_penalty': 1.0,
    'skip_special_tokens': False},
   'prompt_files': {'primer'

### Patients

In [47]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [48]:
embedded_patients.head()

,patient_id,embedding
0,11609,"[0.004150390625, 0.0103759765625, -0.004791259..."
1,12019,"[-0.0546875, 0.060546875, -0.0078125, 0.034667..."
2,15687,"[0.01611328125, -0.006378173828125, -0.0050964..."
3,16916000000,"[0.06298828125, -0.00194549560546875, -0.00424..."
4,1722,"[0.0189208984375, 0.051513671875, -0.003967285..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [49]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'google/gemma-4-31B-it',
   'reasoning_parser': 'auto',
   'chat_template_kwargs': {'enable_thinking': True},
   'sampling_params': {'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 20,
    'min_p': 0.0,
    'presence_penalty': 1.5,
    'max_tokens': 20000,
    'repetition_penalty': 1.0,
    'skip_special_tokens': False},
   'prompt_files': {'primer'

In [50]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches

Generate ranked trial-space candidates for each patient.


In [51]:
from matchminer_ai.matching import generate_candidate_matches

In [52]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Candidate matches


In [53]:
candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [54]:
candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,11609,NCT06810544-3,0.548974,1
1,11609,NCT06810544-4,0.536818,2
2,11609,NCT03423628-1,0.503569,3
3,11609,NCT06552260-1,0.502846,4
4,11609,NCT05256290-4,0.437098,5


## 5. Match quality check


In [55]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the match-quality checker.


In [56]:
candidate_matches_with_context = (
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,11609,NCT06810544-3,0.548974,1,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: >=18 years. Sex allowed: Bo...
1,11609,NCT06810544-4,0.536818,2,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: >=18 years. Sex allowed: Bo...
2,11609,NCT03423628-1,0.503569,3,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: NA. Sex allowed: Both. Canc...
3,11609,NCT06552260-1,0.502846,4,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: >=18 years. Sex allowed: Bo...
4,11609,NCT05256290-4,0.437098,5,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: 18 years or older. Sex allo...


Run the match-quality checker.


In [65]:
match_quality_matches, metadata = score_match_quality(
    candidate_matches_with_context, return_metadata=True
)

In [ ]:
match_quality_matches.head()

,patient_id,space_trial_id,match_quality_score,match_quality_pass
0,11609,NCT06810544-3,0.998111,True
1,11609,NCT06810544-4,0.995954,True
2,11609,NCT03423628-1,0.776640,True
3,11609,NCT06552260-1,0.991066,True
4,11609,NCT05256290-4,0.603532,True


In [ ]:
metadata

## 6. Exclusion criteria check


In [59]:
# limit to unique patient-trial ID pairs
match_quality_matches["trial_id"] = (
    match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,11609,NCT06810544,ECOG 4. Essential hypertension (grade 3 during...,Pregnant or breastfeeding female patients.\nIm...
1,11609,NCT03423628,ECOG 4. Essential hypertension (grade 3 during...,History of severe brain-injury or stroke.\nIne...
2,11609,NCT06552260,ECOG 4. Essential hypertension (grade 3 during...,Poor performance status (Karnofsky Performance...
3,11609,NCT05256290,ECOG 4. Essential hypertension (grade 3 during...,History of interstitial lung disease related t...
4,11609,NCT05739942,ECOG 4. Essential hypertension (grade 3 during...,Concurrent or active therapy for glioblastoma ...


In [60]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 5064.88it/s]


,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,11609,NCT06810544,1.000000,False
1,11609,NCT03423628,0.998710,False
2,11609,NCT06552260,0.999943,False
3,11609,NCT05256290,0.999999,False
4,11609,NCT05739942,0.995752,True


In [61]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True     142
False     20
Name: count, dtype: int64

In [62]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'google/gemma-4-31B-it',
   'reasoning_parser': 'auto',
   'chat_template_kwargs': {'enable_thinking': True},
   'sampling_params': {'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 20,
    'min_p': 0.0,
    'presence_penalty': 1.5,
    'max_tokens': 20000,
    'repetition_penalty': 1.0,
    'skip_special_tokens': False},
   'prompt_files': {'primer'

## 7. Save final matches

Create one review table with one row per patient/trial pair. Each pair is represented by its top-scoring clinical space, rows are ranked by that top match-quality score, and exclusion-check results are included as labels without filtering rows.


In [63]:
patient_context_cols = [
    col
    for col in [
        "patient_id",
        "cancer_history_summary",
        "general_exclusion_criteria_evidence",
    ]
    if col in patient_summaries.columns
]
trial_context_cols = [
    col
    for col in [
        "space_trial_id",
        "clinical_space_number",
        "clinical_space_summary",
        "general_exclusion_criteria",
    ]
    if col in trial_results.columns
]
original_trial_cols = [
    col
    for col in [
        "trial_id",
        "oncore_id",
        "trial_title",
        "brief_summary",
        "detailed_summary",
        "eligibility_criteria",
    ]
    if col in trials.columns
]

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score", "rank"]],
        on=["patient_id", "space_trial_id"],
        how="left",
    )
    .merge(
        exclusion_results,
        on=["patient_id", "trial_id"],
        how="left",
    )
    .merge(
        patient_summaries[patient_context_cols],
        on="patient_id",
        how="left",
    )
    .merge(
        trial_results[trial_context_cols].drop_duplicates("space_trial_id"),
        on="space_trial_id",
        how="left",
    )
    .merge(
        trials[original_trial_cols].drop_duplicates("trial_id"),
        on="trial_id",
        how="left",
    )
)

# Keep one row per patient/trial pair, represented by the top-scoring clinical space.
final_matches = (
    final_matches.sort_values(
        ["patient_id", "trial_id", "match_quality_score"],
        ascending=[True, True, False],
    )
    .drop_duplicates(["patient_id", "trial_id"], keep="first")
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches["match_quality_rank"] = final_matches.groupby("patient_id").cumcount() + 1

final_matches = final_matches.rename(
    columns={
        "match_quality_rank": "match_rank",
        "exclusion_criteria_pass": "passes_exclusion_criteria",
        "cancer_history_summary": "patient_summary",
        "general_exclusion_criteria_evidence": "patient_exclusion_evidence",
        "brief_summary": "trial_brief_summary",
        "detailed_summary": "trial_detailed_summary",
        "eligibility_criteria": "trial_eligibility_criteria",
        "clinical_space_summary": "matched_clinical_space_summary",
        "general_exclusion_criteria": "extracted_trial_exclusion_criteria",
    }
)

ordered_cols = [
    "patient_id",
    "trial_id",
    "match_rank",
    "passes_exclusion_criteria",
    "patient_summary",
    "patient_exclusion_evidence",
    "trial_title",
    "trial_brief_summary",
    "trial_eligibility_criteria",
    "clinical_space_number",
    "matched_clinical_space_summary",
    "extracted_trial_exclusion_criteria",
]
final_matches = final_matches[[col for col in ordered_cols if col in final_matches.columns]]
final_matches.head()


,patient_id,trial_id,match_rank,passes_exclusion_criteria,patient_summary,patient_exclusion_evidence,trial_title,trial_brief_summary,trial_eligibility_criteria,clinical_space_number,matched_clinical_space_summary,extracted_trial_exclusion_criteria
0,11609,NCT06810544,1,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Phase 1/2, Multicenter, Open-Label Study to ...",This is a first in human study of TNG456 alone...,Inclusion Criteria:\n\n* Has a tumor with a co...,3,Age range allowed: >=18 years. Sex allowed: Bo...,Pregnant or breastfeeding female patients.\nIm...
1,11609,NCT06552260,2,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,A Surgical Window of Opportunity Clinical Tria...,This research study is studying troriluzole as...,Inclusion Criteria:\n\n* Age ≥18 years\n* Hist...,1,Age range allowed: >=18 years. Sex allowed: Bo...,Poor performance status (Karnofsky Performance...
2,11609,NCT06287463,3,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Master Protocol for the Multi-Cohort, Open-L...","This is a multicenter, Phase 1/2 clinical tria...",Inclusion Criteria:\n\nGeneral Inclusion Crite...,6,Age range allowed: NA. Sex allowed: Both. Canc...,"Poor performance status (e.g., ECOG > 1)\nPreg..."
3,11609,NCT04557449,4,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A PHASE 1/2A STUDY EVALUATING THE SAFETY, TOLE...","This is a Phase 1/2A, open label, multicenter,...",Inclusion Criteria\n\n* Part 1: Breast Cancer ...,8,Age range allowed: NA. Sex allowed: Both. Canc...,Poor performance status (eg ECOG >1)\nRenal dy...
4,11609,NCT03423628,5,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Phase I, Multicentre Study to Assess the Saf...",This study will test an investigational drug c...,Inclusion Criteria:\n\n* Provision of formalin...,1,Age range allowed: NA. Sex allowed: Both. Canc...,History of severe brain-injury or stroke.\nIne...


In [64]:
# Optional local export for review or downstream analysis.
final_matches.to_csv("output/final_matches.csv", index=None)